# Amazon Electronics — Tải & Xử lý (TPU-optimized)
**Dataset:** `McAuley-Lab/Amazon-Reviews-2023` (Electronics)

In [1]:
%%capture
!pip install datasets huggingface_hub pyarrow pandas

In [2]:
import os
import json
import warnings
import gc
warnings.filterwarnings('ignore')

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.json as paj
from datasets import load_dataset

CATEGORY   = "Electronics"
HF_REPO    = "McAuley-Lab/Amazon-Reviews-2023"
OUTPUT_DIR = "/kaggle/working/amazon_electronics"
os.makedirs(OUTPUT_DIR, exist_ok=True)

import psutil
ram_gb = psutil.virtual_memory().total / 1024**3
print(f"RAM khả dụng: {ram_gb:.0f} GB")
print(f"PyArrow version: {pa.__version__}")
print(f"Pandas version: {pd.__version__}")

RAM khả dụng: 378 GB
PyArrow version: 23.0.1
Pandas version: 3.0.1


In [3]:
print(f"Đang tải Review data cho '{CATEGORY}'...")

review_hf = load_dataset(
    "json",
    data_files=f"hf://datasets/{HF_REPO}/raw/review_categories/{CATEGORY}.jsonl",
    split="train",
)
print(f"Review records : {len(review_hf):,}")
print(f"   Columns     : {review_hf.column_names}")

Đang tải Review data cho 'Electronics'...


raw/review_categories/Electronics.jsonl:   0%|          | 0.00/22.6G [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Loading dataset shards:   0%|          | 0/34 [00:00<?, ?it/s]

Review records : 43,886,944
   Columns     : ['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']


In [4]:
print("Chuyển Review → PyArrow Table...")
review_table = review_hf.data.table
print(f"Review Arrow Table: {review_table.num_rows:,} rows × {review_table.num_columns} cols")
print(f"   Memory: {review_table.nbytes / 1024**3:.2f} GB")

Chuyển Review → PyArrow Table...
Review Arrow Table: 43,886,944 rows × 10 cols
   Memory: 15.56 GB


In [5]:
print("Chuyển sang Pandas DataFrame...")
review_df = review_table.to_pandas()
print(f"review_df : {review_df.shape}")

Chuyển sang Pandas DataFrame...
review_df : (43886944, 10)


In [6]:
# ── Lọc cơ bản: chỉ giữ rating >= 4, drop duplicate (user, item) ──
review_df = review_df[["user_id", "parent_asin", "rating", "timestamp"]]
review_df = review_df[review_df["rating"] >= 4].reset_index(drop=True)
review_df = review_df.sort_values("timestamp").drop_duplicates(
    subset=["user_id", "parent_asin"], keep="last"
).reset_index(drop=True)

print(f"Sau lọc rating>=4 + dedup: {review_df.shape}")

Sau lọc rating>=4 + dedup: (32996562, 4)


## Strict K-Core Filtering

Lặp lặp lại cho đến khi hội tụ: loại bỏ user có < k interaction **và** item có < k interaction trong mỗi vòng. Điều này đảm bảo tập cuối cùng *thực sự* là k-core (không chỉ một chiều).

In [7]:
def strict_kcore_filter(df: pd.DataFrame, k: int = 5,
                        user_col: str = "user_id",
                        item_col: str = "parent_asin") -> pd.DataFrame:
    """
    Lọc k-core chặt (iterative) theo cả user lẫn item.
    Lặp đến khi không còn user hoặc item nào bị loại.
    """
    iteration = 0
    while True:
        n_before = len(df)

        # Loại user có ít hơn k tương tác
        user_counts = df[user_col].value_counts()
        df = df[df[user_col].isin(user_counts[user_counts >= k].index)]

        # Loại item có ít hơn k tương tác
        item_counts = df[item_col].value_counts()
        df = df[df[item_col].isin(item_counts[item_counts >= k].index)]

        iteration += 1
        n_after = len(df)
        dropped = n_before - n_after
        print(f"  Vòng {iteration:2d}: {n_after:,} records | loại {dropped:,} | "
              f"users={df[user_col].nunique():,} | items={df[item_col].nunique():,}")

        if dropped == 0:
            break

    return df.reset_index(drop=True)


K_CORE = 5
print(f"Áp dụng strict {K_CORE}-core filter (user + item, iterative)...")
review_df = strict_kcore_filter(review_df, k=K_CORE)

print(f"\n✓ Kết quả cuối:")
print(f"   Records : {len(review_df):,}")
print(f"   Users   : {review_df['user_id'].nunique():,}")
print(f"   Items   : {review_df['parent_asin'].nunique():,}")
gc.collect()

Áp dụng strict 5-core filter (user + item, iterative)...
  Vòng  1: 11,291,817 records | loại 21,704,745 | users=1,328,082 | items=299,348
  Vòng  2: 10,596,366 records | loại 695,451 | users=1,155,762 | items=284,937
  Vòng  3: 10,554,858 records | loại 41,508 | users=1,146,204 | items=284,067
  Vòng  4: 10,552,057 records | loại 2,801 | users=1,145,558 | items=284,011
  Vòng  5: 10,551,881 records | loại 176 | users=1,145,517 | items=284,008
  Vòng  6: 10,551,877 records | loại 4 | users=1,145,516 | items=284,008
  Vòng  7: 10,551,877 records | loại 0 | users=1,145,516 | items=284,008

✓ Kết quả cuối:
   Records : 10,551,877
   Users   : 1,145,516
   Items   : 284,008


24

## Train / Val / Test Split (Leave-One-Out)

Dùng **leave-one-out theo timestamp**:
- `test`  → interaction mới nhất của mỗi user  
- `val`   → interaction thứ hai từ cuối  
- `train` → phần còn lại  

Sau đó **lọc val/test** để chỉ giữ (user, item) đều đã xuất hiện trong train (tránh cold-start ảnh hưởng kết quả đánh giá).

In [8]:
print("Tạo train/val/test split (leave-one-out theo timestamp)...")

review_sorted = review_df.sort_values(["user_id", "timestamp"])
review_sorted["_rank"] = (
    review_sorted.groupby("user_id").cumcount(ascending=False)
)  # 0 = last, 1 = second-to-last, ...

test_df  = review_sorted[review_sorted["_rank"] == 0].drop(columns="_rank").reset_index(drop=True)
val_df   = review_sorted[review_sorted["_rank"] == 1].drop(columns="_rank").reset_index(drop=True)
train_df = review_sorted[review_sorted["_rank"] >= 2].drop(columns="_rank").reset_index(drop=True)

print(f"Train : {len(train_df):,} ({len(train_df)/len(review_df)*100:.1f}%)")
print(f"Val   : {len(val_df):,}  ({len(val_df)/len(review_df)*100:.1f}%)")
print(f"Test  : {len(test_df):,}  ({len(test_df)/len(review_df)*100:.1f}%)")
gc.collect()

Tạo train/val/test split (leave-one-out theo timestamp)...
Train : 8,260,845 (78.3%)
Val   : 1,145,516  (10.9%)
Test  : 1,145,516  (10.9%)


0

## Lọc Val/Test: Chỉ Giữ User & Item Có Trong Train

LightGCN chỉ có embedding cho các entity xuất hiện trong tập train. Bất kỳ user hoặc item nào không có trong train sẽ không được embed → phải loại khỏi val/test trước khi đánh giá.

In [9]:
# Tập hợp user và item hợp lệ (xuất hiện trong train)
train_users = set(train_df["user_id"])
train_items = set(train_df["parent_asin"])

def filter_split(df: pd.DataFrame, valid_users: set, valid_items: set,
                 split_name: str) -> pd.DataFrame:
    n_before = len(df)
    df = df[
        df["user_id"].isin(valid_users) &
        df["parent_asin"].isin(valid_items)
    ].reset_index(drop=True)
    n_after = len(df)
    print(f"{split_name}: {n_before:,} → {n_after:,} "
          f"(loại {n_before - n_after:,} cold-start records | "
          f"{(n_before-n_after)/n_before*100:.1f}%)")
    return df


val_df  = filter_split(val_df,  train_users, train_items, "Val ")
test_df = filter_split(test_df, train_users, train_items, "Test")

print(f"\n✓ Sau lọc cold-start:")
print(f"   Train users : {train_df['user_id'].nunique():,}")
print(f"   Train items : {train_df['parent_asin'].nunique():,}")
print(f"   Val  users  : {val_df['user_id'].nunique():,}")
print(f"   Test users  : {test_df['user_id'].nunique():,}")

Val : 1,145,516 → 1,143,887 (loại 1,629 cold-start records | 0.1%)
Test: 1,145,516 → 1,141,932 (loại 3,584 cold-start records | 0.3%)

✓ Sau lọc cold-start:
   Train users : 1,145,516
   Train items : 283,184
   Val  users  : 1,143,887
   Test users  : 1,141,932


## Index Mapping cho LightGCN

LightGCN cần **user_idx** và **item_idx** liên tục bắt đầu từ 0.  
Item index được offset bởi `n_users` để xây đồ thị bipartite trên một không gian node chung.

```
 node_id = user_idx          nếu là user  (0 .. n_users-1)
 node_id = n_users + item_idx nếu là item  (n_users .. n_users+n_items-1)
```

In [10]:
# ── Xây mapping từ raw id sang integer index ──
all_users = sorted(train_df["user_id"].unique())
all_items = sorted(train_df["parent_asin"].unique())

user2idx = {u: i for i, u in enumerate(all_users)}
item2idx = {it: i for i, it in enumerate(all_items)}

n_users = len(user2idx)
n_items = len(item2idx)

print(f"n_users : {n_users:,}")
print(f"n_items : {n_items:,}")
print(f"n_nodes (bipartite graph) : {n_users + n_items:,}")


def apply_mapping(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["user_idx"] = df["user_id"].map(user2idx)
    df["item_idx"] = df["parent_asin"].map(item2idx)
    # node_id trong đồ thị bipartite thống nhất
    df["user_node"] = df["user_idx"]
    df["item_node"] = df["item_idx"] + n_users
    return df


train_df = apply_mapping(train_df)
val_df   = apply_mapping(val_df)
test_df  = apply_mapping(test_df)

print("\nMẫu train_df với index:")
print(train_df[["user_id", "user_idx", "user_node",
                "parent_asin", "item_idx", "item_node", "rating"]].head(5))

n_users : 1,145,516
n_items : 283,184
n_nodes (bipartite graph) : 1,428,700

Mẫu train_df với index:
                        user_id  user_idx  user_node parent_asin  item_idx  \
0  AE22236AFRRSMQIKGG7TPTB75QEA         0          0  B0007MGG2M      5181   
1  AE22236AFRRSMQIKGG7TPTB75QEA         0          0  B000XT3L7W     12346   
2  AE22236AFRRSMQIKGG7TPTB75QEA         0          0  B00297IPUY     20107   
3  AE22236AFRRSMQIKGG7TPTB75QEA         0          0  B000JREMTO      8992   
4  AE22236AFRRSMQIKGG7TPTB75QEA         0          0  B005UP82KU     42907   

   item_node  rating  
0    1150697     5.0  
1    1157862     5.0  
2    1165623     5.0  
3    1154508     5.0  
4    1188423     5.0  


## Kiểm Tra Tính Nhất Quán

In [11]:
# Không có NaN trong user_idx / item_idx (tức là mọi entity đều có mapping)
assert train_df["user_idx"].isna().sum() == 0, "Train có user không có mapping!"
assert train_df["item_idx"].isna().sum() == 0, "Train có item không có mapping!"
assert val_df["user_idx"].isna().sum()   == 0, "Val có cold-start user!"
assert val_df["item_idx"].isna().sum()   == 0, "Val có cold-start item!"
assert test_df["user_idx"].isna().sum()  == 0, "Test có cold-start user!"
assert test_df["item_idx"].isna().sum()  == 0, "Test có cold-start item!"

# user_idx và item_idx trong range hợp lệ
assert train_df["user_idx"].between(0, n_users - 1).all()
assert train_df["item_idx"].between(0, n_items - 1).all()

# Mỗi user trong val/test phải có ít nhất 1 interaction trong train
val_test_users = set(val_df["user_id"]) | set(test_df["user_id"])
assert val_test_users.issubset(train_users), "Có user trong val/test không có trong train!"

print("✓ Tất cả assertion đều pass — dữ liệu nhất quán cho LightGCN.")

✓ Tất cả assertion đều pass — dữ liệu nhất quán cho LightGCN.


## Thống Kê Tổng Hợp

In [12]:
total = len(train_df) + len(val_df) + len(test_df)
density = total / (n_users * n_items)

print("=" * 55)
print("THỐNG KÊ DATASET CUỐI")
print("=" * 55)
print(f"  Train     : {len(train_df):>12,} interactions")
print(f"  Val       : {len(val_df):>12,} interactions")
print(f"  Test      : {len(test_df):>12,} interactions")
print(f"  Total     : {total:>12,} interactions")
print(f"  Users     : {n_users:>12,}")
print(f"  Items     : {n_items:>12,}")
print(f"  Density   : {density:>12.6f}  ({density*100:.4f}%)")
print(f"  Avg interactions/user: {total/n_users:.1f}")
print(f"  Avg interactions/item: {total/n_items:.1f}")
print("=" * 55)

print("\nPhân phối số interaction/user (train):")
user_interaction_count = train_df.groupby("user_idx").size()
print(user_interaction_count.describe().round(2))

THỐNG KÊ DATASET CUỐI
  Train     :    8,260,845 interactions
  Val       :    1,143,887 interactions
  Test      :    1,141,932 interactions
  Total     :   10,546,664 interactions
  Users     :    1,145,516
  Items     :      283,184
  Density   :     0.000033  (0.0033%)
  Avg interactions/user: 9.2
  Avg interactions/item: 37.2

Phân phối số interaction/user (train):
count    1145516.00
mean           7.21
std            8.28
min            3.00
25%            3.00
50%            5.00
75%            8.00
max          778.00
dtype: float64


## Xuất File

Các file được xuất:
| File | Mô tả |
|------|-------|
| `train.parquet` | Tập train với user_idx, item_idx, user_node, item_node |
| `val.parquet` | Tập val đã lọc cold-start (user + item) |
| `test.parquet` | Tập test đã lọc cold-start (user + item) |
| `id_mappings.json` | Gộp user2idx + item2idx + n_users + n_items |
| `dataset_info.json` | Metadata: n_users, n_items, n_nodes, splits sizes |


In [13]:
# ─────────────────── Xuất Parquet ───────────────────
train_df.to_parquet(f"{OUTPUT_DIR}/train.parquet", index=False)
val_df.to_parquet(f"{OUTPUT_DIR}/val.parquet", index=False)
test_df.to_parquet(f"{OUTPUT_DIR}/test.parquet", index=False)

print("✓ Đã xuất train/val/test.parquet")


# ─────────────── Xuất id_mappings.json ───────────────
id_mappings = {
    "user2idx": user2idx,
    "item2idx": item2idx,
    "n_users": n_users,
    "n_items": n_items,
}

with open(f"{OUTPUT_DIR}/id_mappings.json", "w") as f:
    json.dump(id_mappings, f, ensure_ascii=False)

print("✓ Đã xuất id_mappings.json")


# ─────────────────── Xuất metadata ───────────────────
dataset_info = {
    "category": CATEGORY,
    "k_core": K_CORE,
    "n_users": n_users,
    "n_items": n_items,
    "n_nodes": n_users + n_items,
    "n_train": len(train_df),
    "n_val": len(val_df),
    "n_test": len(test_df),
    "n_total": total,
    "density": round(density, 8),

    # item node offset trong đồ thị bipartite
    "item_node_offset": n_users,

    # để load lại nhanh trong training script
    "user_col": "user_idx",
    "item_col": "item_idx",
    "user_node_col": "user_node",
    "item_node_col": "item_node",
}

with open(f"{OUTPUT_DIR}/dataset_info.json", "w") as f:
    json.dump(dataset_info, f, indent=2, ensure_ascii=False)

print("✓ Đã xuất dataset_info.json")


# ─────────────── In danh sách file output ───────────────
print()
print("Files trong output directory:")

for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    size_mb = os.path.getsize(fpath) / 1024**2
    print(f"  {fname:<30} {size_mb:8.1f} MB")

✓ Đã xuất train/val/test.parquet
✓ Đã xuất id_mappings.json
✓ Đã xuất dataset_info.json

Files trong output directory:
  dataset_info.json                   0.0 MB
  id_mappings.json                   49.6 MB
  test.parquet                       63.4 MB
  train.parquet                     253.4 MB
  val.parquet                        63.9 MB
